In [1]:
import os
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())
openai_api_key = os.environ["OPENAI_API_KEY"]

In [2]:
from langchain_openai import ChatOpenAI

model = ChatOpenAI(model="gpt-3.5-turbo-0125")

In [3]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

In [4]:
prompt = ChatPromptTemplate.from_template("tell me a curious fact about {soccer_player}")

output_parser = StrOutputParser()

In [6]:
chain = prompt | model | output_parser

chain.invoke({"soccer_player": "Ronaldo"})

'One curious fact about Cristiano Ronaldo is that he has his own museum dedicated to his career in his hometown of Funchal, Madeira. The museum showcases memorabilia, awards, and milestones from his illustrious football career.'

## Use of .bind() to add arguments to a Runnable in a LCEL Chain
* For example, we can add an argument to stop the model response when it reaches the word "Ronaldo":
* `.bind()` is useful when we want to customize a runnable without changing the base prompt or chain structure.
* In this case, we tell the model to stop generating once it reaches the word `Ronaldo`, which helps demonstrate how runtime parameters can be injected into the model call.

In [8]:
chain = prompt | model.bind(stop=["Ronaldo"]) | output_parser
chain.invoke({"soccer_player": "Ronaldo"})

'One curious fact about Cristiano '

# Combining LCEL Chains
This section shows that one runnable chain can be used as a building block inside another chain. That is the real power of LCEL: you can compose smaller steps into more complex workflows without leaving the same pipeline abstraction.

In [9]:
from langchain_openai import ChatOpenAI

model = ChatOpenAI(model="gpt-3.5-turbo")

In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

# First, define a prompt that asks for a sentence about a politician.
prompt = ChatPromptTemplate.from_template("tell me a sentence about {politician}")

# Build a simple chain: prompt -> model -> output parser.
chain = prompt | model | StrOutputParser()

In [ ]:
chain.invoke("Chamberlain")

# Combined chain
Here, the output of one chain becomes the input to another prompt. This pattern is useful when you want to take a generated answer and feed it into a second LLM step for deeper reasoning or validation.

In [ ]:
# A second prompt evaluates the politician using the first chain's output as its input.
historian_prompt = ChatPromptTemplate.from_template("Was {politician} positive for Humanity?")

composed_chain = {"politician": chain} | historian_prompt | model | StrOutputParser()

In [13]:
composed_chain.invoke({"politician": "Lincoln"})

"Yes, Abraham Lincoln is considered positive for humanity because of his leadership during a critical time in American history. He played a key role in preserving the Union and ultimately ending slavery, which was a significant step towards equality and justice for all people. Lincoln's actions helped to shape the future of the United States and have a lasting impact on human rights and liberty."

## Another example: a chain inside another chain
This example demonstrates nested composition. The first chain answers a question like "what country is this politician from?" and that answer is then passed into a second prompt that asks for a higher-level fact about that country.


In [ ]:
from operator import itemgetter

from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

# Prompt 1 asks for the politician's country.
prompt1 = ChatPromptTemplate.from_template("what is the country {politician} is from?")

# Prompt 2 uses that country to ask a second, related question.
prompt2 = ChatPromptTemplate.from_template(
    "what continent is the country {country} in? respond in {language}"
)

model = ChatOpenAI()

chain1 = prompt1 | model | StrOutputParser()

# This chain takes the answer from chain1 and feeds it into prompt2.
chain2 = (
    {"country": chain1, "language": itemgetter("language")}
    | prompt2
    | model
    | StrOutputParser()
)

chain2.invoke({"politician": "Miterrand", "language": "French"})

"François Mitterrand était un homme politique et homme d'État français. Il était le président de la France de 1981 à 1995. La France est située en Europe."

# LCEL chain at work in a typical RAG app
This section shows the classic retrieval-augmented generation pattern: the retriever finds relevant documents, the documents are formatted into context, and that context is placed into a prompt for the LLM. This is a real-world LCEL workflow used for grounding answers in external knowledge.

In [15]:
from langchain_openai import ChatOpenAI

model = ChatOpenAI(model="gpt-3.5-turbo-0125")

In [ ]:
import bs4
from langchain import hub
from langchain_chroma import Chroma
from langchain_community.document_loaders import WebBaseLoader
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Load a web article and extract the main content sections.
loader = WebBaseLoader(
    web_paths=("https://lilianweng.github.io/posts/2023-06-23-agent/",),
    bs_kwargs=dict(
        parse_only=bs4.SoupStrainer(
            class_=("post-content", "post-title", "post-header")
        )
    ),
)

docs = loader.load()

# Split the long content into smaller chunks for better retrieval.
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
splits = text_splitter.split_documents(docs)

vectorstore = Chroma.from_documents(documents=splits, embedding=OpenAIEmbeddings())
retriever = vectorstore.as_retriever()

# Pull a standard RAG prompt template from LangChain Hub.
prompt = hub.pull("rlm/rag-prompt")

def format_docs(docs):
    # Convert retrieved documents into a single text block for the prompt.
    return "\n\n".join(doc.page_content for doc in docs)

# This is the classic RAG chain: retrieve, format, prompt, model, parse.
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | model
    | StrOutputParser()
)

rag_chain.invoke("What is Task Decomposition?")


'Task decomposition is a technique used to break down complex tasks into smaller and simpler steps. This approach helps agents to plan and execute tasks more effectively by dividing them into manageable components. It can be done through prompting techniques like Chain of Thought or Tree of Thoughts, or with task-specific instructions.'